# Business Analysis Validation

This notebook validates the analytical logic used in the project.

Purpose:
- build order-level metrics from the processed item-level dataset
- verify KPI definitions
- confirm that business metrics are not biased by item-row weighting

This notebook is not intended to replace the dashboard.
It is used to document the analytical reasoning behind the dashboard metrics.

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/processed/clean_olist_data.csv")
df["order_purchase_timestamp"] = pd.to_datetime(df["order_purchase_timestamp"])

print("Processed rows:", len(df))
print("Unique orders:", df["order_id"].nunique())

Processed rows: 112650
Unique orders: 98666


In [3]:
order_level = (
    df.sort_values(["order_id", "order_item_id"])
    .groupby("order_id", as_index=False)
    .agg(
        order_purchase_timestamp=("order_purchase_timestamp", "first"),
        order_month=("order_month", "first"),
        order_weekday=("order_weekday", "first"),
        customer_unique_id=("customer_unique_id", "first"),
        order_status=("order_status", "first"),
        is_delivered=("is_delivered", "max"),
        order_revenue=("revenue", "sum"),
        review_score=("review_score", "first"),
        is_late_delivery=("is_late_delivery", "max"),
        delivery_status=("delivery_status", "first"),
        delivery_delay_days=("delivery_delay_days", "first"),
    )
)

order_level.head()

,order_id,order_purchase_timestamp,order_month,order_weekday,customer_unique_id,order_status,is_delivered,order_revenue,review_score,is_late_delivery,delivery_status,delivery_delay_days
0,00010242fe8c5a6d1ba2dd792cb16214,2017-09-13 08:59:02,2017-09,Wednesday,871766c5855e863f6eccc05f988b23cb,delivered,1,72.19,5.0,0,on_time_or_early,-9.0
1,00018f77f2f0320c557190d7a144bdd3,2017-04-26 10:53:06,2017-04,Wednesday,eb28e67c4c0b83846050ddfb8a35d051,delivered,1,259.83,4.0,0,on_time_or_early,-3.0
2,000229ec398224ef6ca0657da4fc703e,2018-01-14 14:33:31,2018-01,Sunday,3818d81c6709e39d06b2738a8d3a2474,delivered,1,216.87,5.0,0,on_time_or_early,-14.0
3,00024acbcdf0a6daa1e931b038114c75,2018-08-08 10:00:35,2018-08,Wednesday,af861d436cfc08b2c2ddefd0ba074622,delivered,1,25.78,4.0,0,on_time_or_early,-6.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,2017-02-04 13:57:51,2017-02,Saturday,64b576fb70d441e8f1b2d7d446e483c5,delivered,1,218.04,5.0,0,on_time_or_early,-16.0


In [4]:
delivered_orders = order_level[order_level["is_delivered"] == 1].copy()

print("Delivered orders:", delivered_orders["order_id"].nunique())
print("Delivered revenue:", round(delivered_orders["order_revenue"].sum(), 2))
print("Average order value:", round(delivered_orders["order_revenue"].mean(), 2))
print("Average review score:", round(delivered_orders["review_score"].mean(), 4))
print("Late delivery rate:", round(delivered_orders["is_late_delivery"].mean(), 4))

Delivered orders: 96478
Delivered revenue: 15419773.75
Average order value: 159.83
Average review score: 4.1427
Late delivery rate: 0.0677


In [5]:
monthly_kpi = (
    delivered_orders.groupby("order_month", as_index=False)
    .agg(
        total_orders=("order_id", "nunique"),
        total_revenue=("order_revenue", "sum"),
        avg_order_value=("order_revenue", "mean"),
        avg_review_score=("review_score", "mean"),
        late_delivery_rate=("is_late_delivery", "mean"),
    )
    .sort_values("order_month")
)

monthly_kpi.head()

,order_month,total_orders,total_revenue,avg_order_value,avg_review_score,late_delivery_rate
0,2016-09,1,143.46,143.460000,1.000000,1.000000
1,2016-10,265,46490.66,175.436453,3.975472,0.007547
2,2016-12,1,19.62,19.620000,5.000000,0.000000
3,2017-01,750,127482.37,169.976493,4.184667,0.029333
4,2017-02,1653,271239.32,164.089123,4.189958,0.029643


In [6]:
category_perf = (
    df[df["is_delivered"] == 1]
    .groupby("product_category_name_english", as_index=False)
    .agg(
        total_revenue=("revenue", "sum"),
        total_orders=("order_id", "nunique"),
        avg_item_revenue=("revenue", "mean"),
    )
    .sort_values("total_revenue", ascending=False)
    .head(10)
)

category_perf

,product_category_name_english,total_revenue,total_orders,avg_item_revenue
43,health_beauty,1412089.53,8647,149.190653
71,watches_gifts,1264333.12,5495,215.793330
7,bed_bath_table,1225209.26,9272,111.860610
65,sports_leisure,1118256.91,7530,132.636331
15,computers_accessories,1032723.77,6530,135.102534
39,furniture_decor,880329.92,6307,107.883569
49,housewares,758392.25,5743,111.610338
20,cool_stuff,691680.89,3559,186.035742
5,auto,669454.75,3810,161.704046
42,garden_tools,567145.68,3448,132.883243


In [7]:
review_delivery = (
    delivered_orders.groupby("delivery_status", as_index=False)
    .agg(
        total_orders=("order_id", "nunique"),
        avg_review_score=("review_score", "mean"),
        avg_delay_days=("delivery_delay_days", "mean"),
        avg_order_value=("order_revenue", "mean"),
    )
    .sort_values("total_orders", ascending=False)
)

review_delivery

,delivery_status,total_orders,avg_review_score,avg_delay_days,avg_order_value
1,on_time_or_early,89936,4.279729,-13.510263,158.640619
0,late,6534,2.256581,10.620141,176.138985
2,unknown,8,4.500000,NaN,172.365000


In [8]:
weekday_weekend = delivered_orders.copy()
weekday_weekend["day_type"] = weekday_weekend["order_weekday"].apply(
    lambda x: "Weekend" if x in ["Saturday", "Sunday"] else "Weekday"
)

weekday_summary = (
    weekday_weekend.groupby("day_type", as_index=False)
    .agg(
        total_orders=("order_id", "nunique"),
        total_revenue=("order_revenue", "sum"),
        avg_order_value=("order_revenue", "mean"),
        avg_review_score=("review_score", "mean"),
    )
)

weekday_summary

,day_type,total_orders,total_revenue,avg_order_value,avg_review_score
0,Weekday,74288,11906352.38,160.272889,4.143870
1,Weekend,22190,3513421.37,158.333545,4.138906


In [9]:
customer_repeat = (
    delivered_orders.groupby("customer_unique_id", as_index=False)
    .agg(
        total_orders=("order_id", "nunique"),
        total_revenue=("order_revenue", "sum"),
    )
)

customer_repeat["customer_type"] = customer_repeat["total_orders"].apply(
    lambda x: "Repeat" if x > 1 else "One-time"
)

repeat_summary = (
    customer_repeat.groupby("customer_type", as_index=False)
    .agg(
        total_customers=("customer_unique_id", "count"),
        avg_customer_revenue=("total_revenue", "mean"),
    )
)

repeat_summary

,customer_type,total_customers,avg_customer_revenue
0,One-time,90557,160.733972
1,Repeat,2801,308.528190


In [10]:
item_level_review_mean = df["review_score"].mean()
order_level_review_mean = delivered_orders["review_score"].mean()

item_level_late_rate = df["is_late_delivery"].mean()
order_level_late_rate = delivered_orders["is_late_delivery"].mean()

comparison = pd.DataFrame(
    {
        "metric": [
            "avg_review_score",
            "late_delivery_rate",
        ],
        "item_level": [
            item_level_review_mean,
            item_level_late_rate,
        ],
        "order_level_delivered": [
            order_level_review_mean,
            order_level_late_rate,
        ],
    }
)

comparison

,metric,item_level,order_level_delivered
0,avg_review_score,4.016069,4.142729
1,late_delivery_rate,0.064492,0.067725


## Final Note

This notebook exists to validate metric logic, not to duplicate dashboard visuals.

Final modeling rule:
- KPI layer → order-level
- category/product analysis → item-level
- primary revenue reporting → delivered orders only